In [37]:
##### Calculates production totals for each sub-national region in the study
# calculates raster stats of production statistics for totals, crop v livestock, and individual commodities 

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling
import rasterstats

In [38]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# load sub-national geographies
capital_geo = gpd.read_file(f"{cd}/Data/Clean/Capital_stock/subnational_capital_stock_final_GEO.shp")
labor_geo = gpd.read_file(f"{cd}/Data/Clean/Labor/subnational_labor_final_GEO.shp")

# Save paths
capital_path = f"{cd}/Data/Clean/Production/subnational_production_capital_geo.csv"
labor_path = f"{cd}/Data/Clean/Production/subnational_production_labor_geo.csv"


In [39]:
##### Define dict of rasters to use 

raster_dict = {
    # Aggregated production
    "total_production_t"     : f"{cd}/Data/Clean/Production/total_production_tonnes_2020.tif",
    "livestock_production_t" : f"{cd}/Data/Clean/Production/livestock_production_tonnes_2020.tif",
    "crop_production_t"      : f"{cd}/Data/Clean/Production/crop_production_metric_ton_2020.tif",
    # Livestock by species
    "BUF_t"                  : f"{cd}/Data/Clean/Production/buffalo_production_tonnes_2020.tif",
    "CATL_t"                 : f"{cd}/Data/Clean/Production/cattle_production_tonnes_2020.tif",
    "CHK_t"                  : f"{cd}/Data/Clean/Production/chicken_production_tonnes_2020.tif",
    "DCK_t"                  : f"{cd}/Data/Clean/Production/ducks_production_tonnes_2020.tif",
    "GOAT_t"                 : f"{cd}/Data/Clean/Production/goats_production_tonnes_2020.tif",
    "HOR_t"                  : f"{cd}/Data/Clean/Production/horses_production_tonnes_2020.tif",
    "OTH_LV_t"               : f"{cd}/Data/Clean/Production/other_production_tonnes_2020.tif",
    "PIG_t"                  : f"{cd}/Data/Clean/Production/pigs_production_tonnes_2020.tif",
    "SHP_t"                  : f"{cd}/Data/Clean/Production/sheep_production_tonnes_2020.tif",
    # MapSPAM crop production
    "wheat_t"                : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_WHEA_A.tif",
    "rice_t"                 : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_RICE_A.tif",
    "maize_t"                : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_MAIZ_A.tif",
    "barley_t"               : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_BARL_A.tif",
    "small_millet_t"         : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_MILL_A.tif",
    "pearl_millet_t"         : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_PMIL_A.tif",
    "sorghum_t"              : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_SORG_A.tif",
    "other_cereals_t"        : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_OCER_A.tif",
    "potato_t"               : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_POTA_A.tif",
    "sweet_potato_t"         : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_SWPO_A.tif",
    "yams_t"                 : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_YAMS_A.tif",
    "cassava_t"              : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_CASS_A.tif",
    "other_roots_t"          : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_ORTS_A.tif",
    "bean_t"                 : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_BEAN_A.tif",
    "chickpea_t"             : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_CHIC_A.tif",
    "cowpea_t"               : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_COWP_A.tif",
    "pigeon_pea_t"           : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_PIGE_A.tif",
    "lentil_t"               : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_LENT_A.tif",
    "other_pulses_t"         : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_OPUL_A.tif",
    "soybean_t"              : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_SOYB_A.tif",
    "groundnut_t"            : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_GROU_A.tif",
    "coconut_t"              : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_CNUT_A.tif",
    "oilpalm_t"              : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_OILP_A.tif",
    "sunflower_t"            : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_SUNF_A.tif",
    "rapeseed_t"             : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_RAPE_A.tif",
    "sesame_t"               : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_SESA_A.tif",
    "other_oil_crops_t"      : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_OOIL_A.tif",
    "sugarcane_t"            : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_SUGC_A.tif",
    "sugarbeet_t"            : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_SUGB_A.tif",
    "cotton_t"               : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_COTT_A.tif",
    "other_fibre_t"          : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_OFIB_A.tif",
    "arabica_coffee_t"       : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_COFF_A.tif",
    "robusta_coffee_t"       : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_RCOF_A.tif",
    "cocoa_t"                : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_COCO_A.tif",
    "tea_t"                  : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_TEAS_A.tif",
    "tobacco_t"              : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_TOBA_A.tif",
    "banana_t"               : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_BANA_A.tif",
    "plantain_t"             : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_PLNT_A.tif",
    "citrus_t"               : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_CITR_A.tif",
    "other_tropical_fruit_t" : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_TROF_A.tif",
    "temperate_fruit_t"      : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_TEMF_A.tif",
    "tomato_t"               : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_TOMA_A.tif",
    "onion_t"                : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_ONIO_A.tif",
    "other_vegetables_t"     : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_VEGE_A.tif",
    "rubber_t"               : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_RUBB_A.tif",
    "rest_of_crops_t"        : f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production/spam2020_V2r0_global_P_REST_A.tif",
}

In [40]:
##### Combine crop and livestock production into total 

livestock_path = f"{cd}/Data/Clean/Production/livestock_production_tonnes_2020.tif"
crop_path = f"{cd}/Data/Clean/Production/crop_production_metric_ton_2020.tif"
output_path = f"{cd}/Data/Clean/Production/total_production_tonnes_2020.tif"

# resample crop raster to match livestock grid
with rasterio.open(livestock_path) as livestock_src:
    livestock = livestock_src.read(1).astype(np.float64)
    livestock[livestock == livestock_src.nodata] = np.nan
    meta = livestock_src.meta.copy()
    dst_crs = livestock_src.crs
    dst_transform = livestock_src.transform
    dst_shape = livestock_src.shape

with rasterio.open(crop_path) as crop_src:
    crops_aligned = np.full(dst_shape, np.nan, dtype=np.float64)
    reproject(
        source=rasterio.band(crop_src, 1),
        destination=crops_aligned,
        dst_crs=dst_crs,
        dst_transform=dst_transform,
        resampling=Resampling.sum  
    )
    crops_aligned[crops_aligned == crop_src.nodata] = np.nan

# combine rasters
total = np.nansum(np.stack([livestock, crops_aligned]), axis=0)

# mask pixels where both inputs were nodata
all_nodata = np.isnan(livestock) & np.isnan(crops_aligned)
total[all_nodata] = np.nan

meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")
out_arr = total.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(out_arr, 1)

In [41]:
##### Define function to get rasterstats 
# Calculates the sum of raster values within each polygon in a GeoDataFrame and adds the result as a new column

def subnational_production_stats(geo, raster_path, column_name, nodata=-9999):

    with rasterio.open(raster_path) as src:
        raster_crs = src.crs

    # ensure crs match
    gdf_proj = geo.to_crs(raster_crs)

    zonal = rasterstats.zonal_stats(
        gdf_proj,
        raster_path,
        stats="sum",
        nodata=nodata
    )

    geo[column_name] = [z["sum"] for z in zonal]
    return geo


# run for each raster
for column_name, raster_path in raster_dict.items():
    capital_geo = subnational_production_stats(capital_geo, raster_path, column_name)

for column_name, raster_path in raster_dict.items():
    labor_geo = subnational_production_stats(labor_geo, raster_path, column_name)

KeyboardInterrupt: 

In [34]:
### Filter to needed info

col_to_keep = [
    # identifiers
    'ISO3', 'GEO_ID',
    # aggregated production
    'total_production_t', 'livestock_production_t', 'crop_production_t',
    # livestock by species
    'BUF_t', 'CATL_t', 'CHK_t', 'DCK_t', 'GOAT_t', 'HOR_t', 'OTH_LV_t', 'PIG_t', 'SHP_t',
    # MapSPAM crops
    'wheat_t', 'rice_t', 'maize_t', 'barley_t', 'small_millet_t', 'pearl_millet_t',
    'sorghum_t', 'other_cereals_t', 'potato_t', 'sweet_potato_t', 'yams_t', 'cassava_t',
    'other_roots_t', 'bean_t', 'chickpea_t', 'cowpea_t', 'pigeon_pea_t', 'lentil_t',
    'other_pulses_t', 'soybean_t', 'groundnut_t', 'coconut_t', 'oilpalm_t', 'sunflower_t',
    'rapeseed_t', 'sesame_t', 'other_oil_crops_t', 'sugarcane_t', 'sugarbeet_t', 'cotton_t',
    'other_fibre_t', 'arabica_coffee_t', 'robusta_coffee_t', 'cocoa_t', 'tea_t', 'tobacco_t',
    'banana_t', 'plantain_t', 'citrus_t', 'other_tropical_fruit_t', 'temperate_fruit_t',
    'tomato_t', 'onion_t', 'other_vegetables_t', 'rubber_t', 'rest_of_crops_t',
]

capital_geo = capital_geo[col_to_keep]
labor_geo = labor_geo[col_to_keep]

In [ ]:
### Save
capital_geo.to_csv(capital_path, index=False)